In [1]:
import pandas as pd
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [2]:
def add_yoy_growth(df, value_column='value', group_column='root_hs_code', date_column='date'):
    """
    root_hs_code별로 value 컬럼의 연간 증가율을 계산하여 새로운 컬럼으로 추가합니다.
    """
    df = df.copy()
    # top_company_codes = avg_yoy_by_code.sort_values(by='avg_forecast_yoy', ascending=False).head(company_num)
    df = df.dropna(axis=0)
    df[date_column] = pd.to_datetime(df[date_column])
    df.sort_values(by=[group_column, date_column], inplace=True)

    # YoY (12개월 전 대비 비율 변화율) 계산
    df[f'{value_column}_yoy'] = (
        df.groupby(group_column)[value_column]
        .transform(lambda x: x.pct_change(periods=12))
    )

    return df

In [3]:
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host' : '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

trade_df = fetch_table_data(db_info, 'korea_monthly_trade_data_forecast')

import pandas as pd

# date를 datetime으로 변환
trade_df['date'] = pd.to_datetime(trade_df['date'])

# 결측치 제거
trade_df = trade_df.dropna(subset=['expDlr_forecast_12m'])

# root_hs_code, date로 정렬
trade_df = trade_df.sort_values(['root_hs_code', 'date'])

# 그룹 연산 준비
grouped = trade_df.groupby('root_hs_code')

# 이전 12개월 합계 (trailing)
trade_df['export_trail_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum()
)

# 이후 12개월 합계 (forward)
# shift(-11)은 앞으로 11개월 밀어 rolling 12로 보면 해당 시점 기준 이후 12개월을 의미
trade_df['export_forward_12m'] = grouped['expDlr_forecast_12m'].transform(
    lambda x: x.shift(-11).rolling(window=12, min_periods=12).sum()
)

# YoY 성장률
trade_df['export_yoy_growth'] = (
    (trade_df['export_forward_12m'] / trade_df['export_trail_12m']) - 1
)

# 필요한 컬럼만 보기
result_df = trade_df[['date', 'root_hs_code',
                      'export_trail_12m', 'export_forward_12m', 'export_yoy_growth']]

# 필요시 최근 데이터만
trade_yoy_growth = result_df[result_df['date'] == '2025-06-30']

# 확인
# print(trade_yoy_growth.head(20))


✅ 'korea_monthly_trade_data_forecast' 테이블에서 167220건의 데이터를 가져왔습니다.


In [4]:
len(trade_yoy_growth['root_hs_code'].unique().tolist())

753

In [5]:
trade_yoy_growth[trade_yoy_growth['root_hs_code'] == '850213']

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
157696,2025-06-30,850213,433362200.0,786418970.0,0.814692


In [6]:
company_df = fetch_table_data(db_info, 'hs_code_by_kr_monster_company')
company_df.rename(columns={'hs_code_6d': 'root_hs_code'}, inplace=True)

✅ 'hs_code_by_kr_monster_company' 테이블에서 185건의 데이터를 가져왔습니다.


In [7]:
monster_df = pd.merge(company_df, trade_yoy_growth, on='root_hs_code', how='left', indicator=True)

In [12]:
monster_df[monster_df['Code'] == "A004000"]

,hs_code,품목명,Code,Name,root_hs_code,date,export_trail_12m,export_forward_12m,export_yoy_growth,_merge
53,3912399000,셀룰로스 Anycoat,A004000,롯데정밀화학,391239,2025-06-30,368556300.0,404553500.0,0.097671,both


In [9]:
trade_yoy_growth

,date,root_hs_code,export_trail_12m,export_forward_12m,export_yoy_growth
116067,2025-06-30,121120,83255170.0,8.440063e+07,0.013758
116068,2025-06-30,121221,517919500.0,5.963397e+08,0.151414
116069,2025-06-30,151550,12819231.0,8.647523e+06,-0.325426
116070,2025-06-30,151590,5953660.0,6.887209e+06,0.156803
116071,2025-06-30,170199,155429080.0,1.429628e+08,-0.080206
...,...,...,...,...,...
116588,2025-06-30,903180,864757800.0,8.343762e+08,-0.035133
116589,2025-06-30,903190,497127200.0,4.965941e+08,-0.001072
116590,2025-06-30,903289,870932100.0,9.649287e+08,0.107926
116591,2025-06-30,940330,12078255.0,1.106288e+07,-0.084067
